# PoliMillionaire Model Tests

NLP assignment, 2025/26.

| Student | Polimi email |
| --- | --- |
| Juan Martin Sanchez | juanmartin.sancehz@mail.polimi.it |
| Add name | Add email |
| Add name | Add email |

Video: add link here.

Academic integrity: add required statement here.


## Run Order

Pick the local models in `MODELS_TO_TEST`, then run downward.


In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


## Settings

Add or remove models here. They run one at a time.


In [ ]:
INSTALL_DEPS = False

MODELS_TO_TEST = [
    {"label": "Gemma 4 E2B", "kind": "gemma", "model_id": "google/gemma-4-E2B-it", "run": True},
    {"label": "Gemma 4 E4B", "kind": "gemma", "model_id": "google/gemma-4-E4B-it", "run": True},
    {"label": "Qwen3.5 2B Thinking", "kind": "qwen_thinking", "model_id": "Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B Council", "kind": "gemma_council", "model_id": "google/gemma-4-E2B-it", "votes": 3, "run": False},
    {"label": "Gemma + Qwen Mixed Council", "kind": "mixed_council", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
]

print("Install deps:", INSTALL_DEPS)
print("Models:")
for item in MODELS_TO_TEST:
    print("-", item["label"], item["model_id"], "run=", item["run"])


In [ ]:
if INSTALL_DEPS:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")])
else:
    print("Install skipped.")


## Environment

Checks the notebook kernel before loading any model.


In [ ]:
import importlib.util
from packaging.version import Version

print("Python:", sys.executable)

for package_name in ["torch", "transformers", "accelerate"]:
    available = importlib.util.find_spec(package_name) is not None
    version = None
    if available:
        try:
            module = __import__(package_name)
            version = getattr(module, "__version__", "installed")
        except Exception as exc:
            version = f"import error: {exc}"
    print(f"{package_name}: {available} {version or ''}")

if importlib.util.find_spec("transformers") is not None:
    import transformers

    if Version(transformers.__version__) < Version("5.7.0"):
        print("Transformers is too old for the selected local models. Set INSTALL_DEPS=True, run install, restart.")


## Test Question

Same question for every selected model.


In [ ]:
from polimillionaire.types import AnswerOption, Question

sample_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
    level=1,
)

print(sample_question.text)
for option in sample_question.options:
    print(option.id, option.text)


## Run Selected Models

Each model is unloaded before the next one starts.


In [ ]:
import gc

from polimillionaire.strategies import CouncilStrategy, GemmaLLM, GemmaLLMConfig, GemmaStrategy, QwenLLM, QwenLLMConfig, QwenStrategy


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception as exc:
        print("Memory cleanup warning:", exc)


def make_strategy(item):
    if item["kind"] == "mixed_council":
        gemma = GemmaLLM(
            GemmaLLMConfig(
                model_id="google/gemma-4-E2B-it",
                inference_backend="auto_model",
                device_map="auto",
                dtype="auto",
                max_new_tokens=48,
                do_sample=True,
                seed=42,
                generation_max_time_seconds=15.0,
                timeout_seconds=120.0,
            )
        )
        qwen = QwenLLM(
            QwenLLMConfig(
                model_id="Qwen/Qwen3.5-2B",
                device_map="auto",
                dtype="auto",
                max_new_tokens=48,
                temperature=0.7,
                do_sample=True,
                top_p=0.8,
                top_k=20,
                seed=43,
                enable_thinking=False,
                generation_max_time_seconds=15.0,
                timeout_seconds=120.0,
            )
        )
        return CouncilStrategy(
            candidate_llms=[gemma, qwen],
            judge_llm=gemma,
            base_seed=200,
            candidate_max_new_tokens=48,
            judge_max_new_tokens=8,
            max_time_per_call=15.0,
        )
    if item["kind"] == "gemma_council":
        llm = GemmaLLM(
            GemmaLLMConfig(
                model_id=item["model_id"],
                inference_backend="auto_model",
                device_map="auto",
                dtype="auto",
                max_new_tokens=48,
                do_sample=True,
                seed=42,
                generation_max_time_seconds=10.0,
                timeout_seconds=120.0,
            )
        )
        return CouncilStrategy(
            llm=llm,
            num_votes=item.get("votes", 3),
            base_seed=100,
            max_time_per_call=10.0,
        )
    if item["kind"] == "qwen_thinking":
        return QwenStrategy(
            model_config=QwenLLMConfig(
                model_id=item["model_id"],
                device_map="auto",
                dtype="auto",
                max_new_tokens=256,
                temperature=1.0,
                do_sample=True,
                top_p=0.95,
                top_k=20,
                seed=42,
                enable_thinking=True,
                generation_max_time_seconds=60.0,
                timeout_seconds=120.0,
            )
        )
    return GemmaStrategy(
        model_config=GemmaLLMConfig(
            model_id=item["model_id"],
            inference_backend="auto_model",
            device_map="auto",
            dtype="auto",
            max_new_tokens=8,
            temperature=0.0,
            do_sample=False,
            num_beams=1,
            seed=42,
            generation_max_time_seconds=18.0,
            timeout_seconds=120.0,
        )
    )


def run_model_test(item):
    model = None
    try:
        model = make_strategy(item)
        prediction = model.answer(sample_question)

        print()
        print(item["label"])
        print("model:", item["model_id"])
        print("option:", prediction.option_id, prediction.answer_text)
        print("confidence:", prediction.confidence)
        print("reason:", prediction.reasoning)
        print("fallback:", prediction.metadata.get("fallback"))
        print("device:", prediction.metadata.get("device"))
        print("decision:", prediction.metadata.get("decision_method"))
        print("votes:", prediction.metadata.get("votes"))
        print("raw:", prediction.metadata.get("raw_text"))

        if prediction.metadata.get("fallback"):
            raise RuntimeError(f"{item['label']} output was not parsed cleanly.")
        return {
            "label": item["label"],
            "model_id": item["model_id"],
            "option_id": prediction.option_id,
            "answer_text": prediction.answer_text,
            "confidence": prediction.confidence,
            "reason": prediction.reasoning,
            "fallback": prediction.metadata.get("fallback"),
        }
    finally:
        del model
        clear_model_memory()
        print("Cleared model memory after", item["label"])


model_results = []
for item in MODELS_TO_TEST:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue
    model_results.append(run_model_test(item))

model_results


## Saved Logs

Optional check of previous Gemma live runs.


In [ ]:
from polimillionaire.runner import load_jsonl, summarize_attempts

log_dir = REPO_ROOT / "results" / "runs"
gemma_logs = sorted(log_dir.glob("gemma*.jsonl"), key=lambda path: path.stat().st_mtime)

for path in gemma_logs:
    print(path.name)

if gemma_logs:
    rows = load_jsonl(gemma_logs[-1])
    print("latest:", gemma_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No Gemma logs found.")


## Done

Run cleanly, keep useful outputs, export to HTML.
